# 01 — Data Cleaning

## Objective

Reproduce the cleaned inputs from the official source downloads:

- Environment and Climate Change Canada daily climate data for Squamish Airport, 2020
- Water Survey of Canada daily discharge for Squamish River near Brackendale (`08GA022`)

The reusable functions retain only the variables needed for this project, parse dates and numeric values, remove duplicate dates, and validate the expected 2020 coverage. Missing climate observations are preserved here and handled transparently during feature engineering. Streamflow is never interpolated.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from clean_climate import clean_climate
from clean_streamflow import clean_streamflow
from merge_data import load_and_merge

## 1. Clean Squamish Airport climate data

The ECCC download is reduced to date, mean temperature, and total precipitation. The 366 leap-year dates are validated before saving.

In [2]:
climate = clean_climate(persist=True)
climate.head()

Climate rows: 366
date                0
mean_temp_c         4
precipitation_mm    9


,date,mean_temp_c,precipitation_mm
0,2020-01-01,5.8,13.0
1,2020-01-02,2.3,10.8
2,2020-01-03,4.1,15.0
3,2020-01-04,5.5,6.0
4,2020-01-05,3.2,3.2


In [3]:
climate.isna().sum().to_frame("missing_values")

,missing_values
date,0
mean_temp_c,4
precipitation_mm,9


Four mean-temperature values and nine precipitation values are missing in the official daily record. They are not silently replaced during cleaning.

## 2. Clean Squamish River streamflow data

The Water Survey of Canada file contains discharge (`PARAM=1`) and water level (`PARAM=2`). Only discharge for station `08GA022` is retained. The full cleaned station record is saved because it may support future multi-year extensions, while this project merges only dates that overlap the 2020 climate file.

In [4]:
streamflow = clean_streamflow(persist=True)
streamflow_2020 = streamflow.loc[streamflow["date"].dt.year == 2020]
streamflow_2020.head()

Streamflow rows (full station record): 37,659
Verified 366 complete discharge observations for 2020.


,date,streamflow_cms
35467,2020-01-01,135.0
35468,2020-01-02,135.0
35469,2020-01-03,131.0
35470,2020-01-04,234.0
35471,2020-01-05,140.0


In [5]:
streamflow_2020.isna().sum().to_frame("missing_values")

,missing_values
date,0
streamflow_cms,0


## 3. Merge by date

An inner join keeps dates observed by both sources. Because the climate file covers all 366 dates in 2020 and streamflow is complete for those dates, the merged result also contains 366 rows.

In [6]:
merged = load_and_merge(persist=True)
merged.head()

Merged rows: 366
Date range: 2020-01-01 to 2020-12-31
Missing values:
date                0
mean_temp_c         4
precipitation_mm    9
streamflow_cms      0


,date,mean_temp_c,precipitation_mm,streamflow_cms
0,2020-01-01,5.8,13.0,135.0
1,2020-01-02,2.3,10.8,135.0
2,2020-01-03,4.1,15.0,131.0
3,2020-01-04,5.5,6.0,234.0
4,2020-01-05,3.2,3.2,140.0


In [7]:
pd.DataFrame({
    "rows": [len(merged)],
    "start": [merged["date"].min().date()],
    "end": [merged["date"].max().date()],
    "duplicate_dates": [merged["date"].duplicated().sum()],
})

,rows,start,end,duplicate_dates
0,366,2020-01-01,2020-12-31,0


In [8]:
merged.isna().sum().to_frame("missing_values")

,missing_values
date,0
mean_temp_c,4
precipitation_mm,9
streamflow_cms,0


## Output schemas

- `climate_clean.csv`: `date`, `mean_temp_c`, `precipitation_mm`
- `streamflow_clean.csv`: `date`, `streamflow_cms`
- `hydroclimate_merged.csv`: all four variables joined by date

The next notebook explores the merged real dataset before any modeling transformations.